In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import sys

# srcディレクトリをPythonの検索パスに追加
sys.path.append("/home/keiseki/JR_train_snow/30.src")
from utils.utils import plot_feature_vs_target


warnings.filterwarnings("ignore", category=UserWarning)

---
風速計データの確認

In [ ]:
# df_wind = pd.read_csv("/home/keiseki/JR_train_snow/20.Data/wind_location.csv", encoding='cp932', index_col=0)
# df_wind

---
気象データの確認

In [ ]:
df_weather = pd.read_csv("/home/keiseki/JR_train_snow/20.Data/weather.csv", encoding='cp932', index_col=0)
df_weather

---
積雪計データの確認

In [ ]:
df_snow_meter = pd.read_csv("/home/keiseki/JR_train_snow/20.Data/snowfall.csv", encoding='cp932', index_col=0)
df_snow_meter

---


In [ ]:
df_train = pd.read_pickle("/home/keiseki/JR_train_snow/20.Data/train_weather_snow.pkl")
df_test = pd.read_pickle("/home/keiseki/JR_train_snow/20.Data/test_weather_snow.pkl")

In [ ]:
plot_feature_vs_target(df_train, feature_cols=["月", '金沢着雪ゼロ列車フラグ', "気温"], target_col="合計", bins=20, figsize=(8, 5))

In [ ]:
import pandas as pd
import numpy as np


def analyze_snow_rate_by_temperature(
    df,
    temp_col,
    target_col,
    temp_bins=None
):
    """
    気温帯ごとの着雪発生率を集計する。

    Parameters
    ----------
    df : pandas.DataFrame
        分析対象データ
    temp_col : str
        気温の列名
    target_col : str
        目的変数の列名
    temp_bins : list or None
        気温帯の境界。
        Noneの場合は1℃刻み。

    Returns
    -------
    summary : pandas.DataFrame
        気温帯ごとの件数・着雪発生件数・発生率など
    """

    data = df[[temp_col, target_col]].copy()

    # 数値化
    data[temp_col] = pd.to_numeric(
        data[temp_col],
        errors="coerce"
    )

    data[target_col] = pd.to_numeric(
        data[target_col],
        errors="coerce"
    )

    # 欠損除外
    data = data.dropna()

    # 気温帯を指定
    if temp_bins is None:
        temp_min = np.floor(data[temp_col].min())
        temp_max = np.ceil(data[temp_col].max())

        temp_bins = np.arange(
            temp_min,
            temp_max + 1,
            1
        )

    data["気温帯"] = pd.cut(
        data[temp_col],
        bins=temp_bins,
        right=False
    )

    # 着雪発生フラグ
    data["着雪発生"] = data[target_col] > 0

    # 集計
    summary = data.groupby(
        "気温帯",
        observed=True
    ).agg(
        件数=("着雪発生", "size"),
        着雪発生件数=("着雪発生", "sum"),
        平均着雪量=(target_col, "mean"),
        最大着雪量=(target_col, "max")
    )

    # 発生率
    summary["着雪発生率"] = (
        summary["着雪発生件数"]
        / summary["件数"]
        * 100
    )

    return summary

In [ ]:
for col in df_train.columns:
    if "気温" in col:
        print(f"--- {col} ---")

        summary = analyze_snow_rate_by_temperature(
            df_train,
            temp_col=col,
            target_col="合計"
        )

        display(summary)